In [2]:
from __future__ import annotations

import pandas as pd
from pathlib import Path
from typing import Iterable, Literal

AggSide = Literal["start", "end", "both"]
Freq = Literal["H", "D"]  # hourly or daily


def _read_journey_csv(path: str | Path) -> pd.DataFrame:
    """
    Reads one TfL cycle hire journey extract CSV and parses timestamps.
    Assumes columns like:
      - 'Start Date', 'StartStation Id'
      - 'End Date', 'EndStation Id'
      - 'Duration'
    """
    df = pd.read_csv(path)

    # Parse timestamps (day-first because sample is dd/mm/yyyy)
    df["Start Date"] = pd.to_datetime(df["Start Date"], dayfirst=True, errors="coerce")
    df["End Date"] = pd.to_datetime(df["End Date"], dayfirst=True, errors="coerce")

    # Ensure station IDs numeric (some end IDs can be NaN)
    df["StartStation Id"] = pd.to_numeric(df["StartStation Id"], errors="coerce").astype("Int64")
    df["EndStation Id"] = pd.to_numeric(df["EndStation Id"], errors="coerce").astype("Int64")

    # Duration numeric
    if "Duration" in df.columns:
        df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")

    return df


def aggregate_station_time(
    journey_files: Iterable[str | Path],
    *,
    freq: Freq = "H",
    side: AggSide = "start",
) -> pd.DataFrame:
    """
    Aggregates multiple journey CSVs into a station-time panel.

    Returns columns:
      - station_id
      - ts (floored to hour or day)
      - trips_start / trips_end (depending on side)
      - avg_duration_start / avg_duration_end (if Duration available)
    """
    parts: list[pd.DataFrame] = []

    for f in journey_files:
        j = _read_journey_csv(f)

        def agg_one(which: Literal["start", "end"]) -> pd.DataFrame:
            if which == "start":
                ts = j["Start Date"]
                sid = j["StartStation Id"]
                dur = j["Duration"] if "Duration" in j.columns else None
                trips_col = "trips_start"
                dur_col = "avg_duration_start"
            else:
                ts = j["End Date"]
                sid = j["EndStation Id"]
                dur = j["Duration"] if "Duration" in j.columns else None
                trips_col = "trips_end"
                dur_col = "avg_duration_end"

            tmp = pd.DataFrame({"station_id": sid, "ts": ts})
            tmp = tmp.dropna(subset=["station_id", "ts"])
            tmp["station_id"] = tmp["station_id"].astype(int)

            # Floor timestamps to hour/day
            if freq == "H":
                tmp["ts"] = tmp["ts"].dt.floor("H")
            elif freq == "D":
                tmp["ts"] = tmp["ts"].dt.floor("D")
            else:
                raise ValueError("freq must be 'H' or 'D'")

            g = tmp.groupby(["station_id", "ts"], as_index=False).size()
            g = g.rename(columns={"size": trips_col})

            # Optional duration aggregation
            if dur is not None:
                tmp2 = pd.DataFrame({"station_id": sid, "ts": ts, "duration": dur})
                tmp2 = tmp2.dropna(subset=["station_id", "ts", "duration"])
                tmp2["station_id"] = tmp2["station_id"].astype(int)
                tmp2["ts"] = tmp2["ts"].dt.floor("H" if freq == "H" else "D")
                d = tmp2.groupby(["station_id", "ts"], as_index=False)["duration"].mean()
                d = d.rename(columns={"duration": dur_col})
                g = g.merge(d, on=["station_id", "ts"], how="left")

            return g

        if side in ("start", "both"):
            parts.append(agg_one("start"))
        if side in ("end", "both"):
            parts.append(agg_one("end"))

    # Merge all parts (outer join over station_id, ts)
    out = None
    for p in parts:
        out = p if out is None else out.merge(p, on=["station_id", "ts"], how="outer")

    out = out.fillna(0)
    # If duration columns exist, keep them as float with NaNs allowed rather than 0
    for c in ["avg_duration_start", "avg_duration_end"]:
        if c in out.columns:
            out[c] = out[c].replace(0, pd.NA)

    out = out.sort_values(["station_id", "ts"]).reset_index(drop=True)
    return out


# Example usage on your sample file:
# base = aggregate_station_time(
#     journey_files=["/mnt/data/01aJourneyDataExtract10Jan16-23Jan16.csv"],
#     freq="H",
#     side="both",
# )
# print(base.head())


In [ ]:
base = aggregate_station_time(
    journey_files=["/mnt/data/01aJourneyDataExtract10Jan16-23Jan16.csv"],
    freq="H",
    side="both",
)
print(base.head())


In [ ]:
from pathlib import Path
import pandas as pd
from typing import Literal

AggSide = Literal["start", "end", "both"]
Freq = Literal["h", "d"]

def aggregate_from_folder_chunked(
    folder_path: str | Path,
    *,
    freq: Freq = "h",
    side: AggSide = "start",
    chunksize: int = 250_000,   # tune this
) -> pd.DataFrame:
    folder = Path(folder_path)
    files = sorted(folder.rglob("*.csv"))
    if not files:
        raise ValueError("No CSV files found in folder.")
    print(f"Found {len(files)} files.")

    # Running tall table: station_id, ts, trips_start, trips_end
    acc = pd.DataFrame(columns=["station_id", "ts", "trips_start", "trips_end"])

    # Only read what we need (massive memory reduction)
    usecols = []
    if side in ("start", "both"):
        usecols += ["StartStation Id", "Start Date"]
    if side in ("end", "both"):
        usecols += ["EndStation Id", "End Date"]

    # Ensure uniqueness
    usecols = list(dict.fromkeys(usecols))

    for f in files:
        print(f"Processing {f.name} ...")

        for chunk in pd.read_csv(
            f,
            usecols=usecols,
            chunksize=chunksize,
            low_memory=True,
        ):
            out_parts = []

            if side in ("start", "both"):
                tmp = chunk[["StartStation Id", "Start Date"]].copy()
                tmp = tmp.dropna()
                tmp["station_id"] = pd.to_numeric(tmp["StartStation Id"], errors="coerce")
                tmp["ts"] = pd.to_datetime(tmp["Start Date"], dayfirst=True, errors="coerce")
                tmp = tmp.dropna(subset=["station_id", "ts"])
                tmp["station_id"] = tmp["station_id"].astype(int)
                tmp["ts"] = tmp["ts"].dt.floor("H" if freq == "H" else "D")
                g = tmp.groupby(["station_id", "ts"], as_index=False).size()
                g = g.rename(columns={"size": "trips_start"})
                out_parts.append(g)

            if side in ("end", "both"):
                tmp = chunk[["EndStation Id", "End Date"]].copy()
                tmp = tmp.dropna()
                tmp["station_id"] = pd.to_numeric(tmp["EndStation Id"], errors="coerce")
                tmp["ts"] = pd.to_datetime(tmp["End Date"], dayfirst=True, errors="coerce")
                tmp = tmp.dropna(subset=["station_id", "ts"])
                tmp["station_id"] = tmp["station_id"].astype(int)
                tmp["ts"] = tmp["ts"].dt.floor("H" if freq == "H" else "D")
                g = tmp.groupby(["station_id", "ts"], as_index=False).size()
                g = g.rename(columns={"size": "trips_end"})
                out_parts.append(g)

            if not out_parts:
                continue

            chunk_agg = out_parts[0]
            for p in out_parts[1:]:
                chunk_agg = chunk_agg.merge(p, on=["station_id", "ts"], how="outer")

            # Add to accumulator without outer-merge loops:
            # concat tall + groupby sum is far more stable than repeated merges.
            acc = pd.concat([acc, chunk_agg], ignore_index=True)
            acc["trips_start"] = pd.to_numeric(acc.get("trips_start", 0), errors="coerce")
            acc["trips_end"] = pd.to_numeric(acc.get("trips_end", 0), errors="coerce")

            acc = (
                acc.groupby(["station_id", "ts"], as_index=False)[["trips_start", "trips_end"]]
                .sum(min_count=1)
            )

    # Fill missing columns depending on 'side'
    if "trips_start" not in acc.columns:
        acc["trips_start"] = 0
    if "trips_end" not in acc.columns:
        acc["trips_end"] = 0

    acc = acc.sort_values(["station_id", "ts"]).reset_index(drop=True)
    print("Aggregation complete.")
    return acc



In [4]:
base = aggregate_from_folder_chunked(
    r"E:\tfl_project\data",
    freq="H",        # or "D"
    side="start"     # "start", "end", or "both"
)

Found 117 files.
Processing 01aJourneyDataExtract10Jan16-23Jan16.csv ...


ValueError: Invalid frequency: H. Failed to parse with error message: ValueError("Invalid frequency: H. Failed to parse with error message: KeyError('H'). Did you mean h?") Did you mean h?

In [4]:
test = pd.read_csv("E:\\tfl_project\data\\01aJourneyDataExtract10Jan16-23Jan16.csv")

In [5]:
test.head()

,Rental Id,Duration,Bike Id,End Date,EndStation Id,EndStation Name,Start Date,StartStation Id,StartStation Name
0,50754225,240,11834,10/01/2016 00:04,383.0,"Frith Street, Soho",10/01/2016 00:00,18,"Drury Lane, Covent Garden"
1,50754226,300,9648,10/01/2016 00:05,719.0,"Victoria Park Road, Hackney Central",10/01/2016 00:00,479,"Pott Street, Bethnal Green"
2,50754227,1200,10689,10/01/2016 00:20,272.0,"Baylis Road, Waterloo",10/01/2016 00:00,425,"Harrington Square 2, Camden Town"
3,50754228,780,8593,10/01/2016 00:14,471.0,"Hewison Street, Old Ford",10/01/2016 00:01,487,"Canton Street, Poplar"
4,50754229,600,8619,10/01/2016 00:11,399.0,"Brick Lane Market, Shoreditch",10/01/2016 00:01,501,"Cephas Street, Bethnal Green"
